# Delta Lake Training Notebook

Hands-on SQL guide to Delta Lake using the **TPC-H benchmark dataset** (7.5 M orders, 750 K customers) from the shared `samples.tpch` catalog. **All exercises are pure SQL; no PySpark required.**

| # | Topic | What you will learn |
|---|---|---|
| 1 | Create | CTAS, DDL, INSERT |
| 2 | DML | UPDATE, DELETE, MERGE |
| 3 | Storage, Delta Log & Checkpoints | File layout, commit JSON files, checkpoints, DESCRIBE HISTORY |
| 4 | Time Travel | VERSION AS OF, TIMESTAMP AS OF, @v syntax, RESTORE |
| 5 | Change Data Feed | delta.enableChangeDataFeed, table_changes() |
| 6 | Constraints | NOT NULL, CHECK, PRIMARY / FOREIGN KEY |
| 7 | Indexes & Data Skipping | File statistics, Bloom filter indexes, ANALYZE |
| 8 | Performance Techniques | Partitioning, Z-Order, Liquid Clustering, compaction, optimized writes, predictive optimization, key table properties |
| 9 | VACUUM | Retention, DRY RUN, LITE vs FULL |

**Working tables** (created in `one_dbx_dev.default`):
- `tpc_orders` — main table for DML, versions, and time travel
- `tpc_orders_partitioned` — partitioned by order year
- `tpc_orders_clustered` — managed by Liquid Clustering
- `tpc_log_demo` — tiny table to illustrate commits and checkpoints
- `tpc_orders_cdf` — Change Data Feed, constraints and indexes
- `tpc_customer` — parent table for primary / foreign keys
- `tpc_orders_small` — small-file problem, compaction and optimized writes

##DataLake
* Need for an organization to collect data from various sources of structured/ semistructured and unstructured data types
* Usage of these infromations in Bigdata and datascience applications
* Scalabilty and cost effective
<img src="/Volumes/training/bronze/images/Screenshot 2026-10-17 at 16.22.04.png" alt="Data Lake Sources" width="800"/>

##DELTALAKE = Datalake + ACID (Datalog and Schema Enforcement ​)

* A — ATOMICITY​
DELTA LAKE WRITES ALL FILE CHANGES AS A SINGLE ATOMIC COMMIT IN THE _DELTA_LOG, SO EITHER THE ENTIRE TRANSACTION SUCCEEDS OR NOTHING IS APPLIED.​
* C — CONSISTENCY​
SCHEMA ENFORCEMENT AND CONSTRAINTS ENSURE EVERY COMMIT RECORDED IN THE _DELTA_LOG KEEPS THE TABLE IN A VALID, CONSISTENT STATE.​
* I — ISOLATION​
 USING CONCURRENCY CONTROL EACH WRITER/READER SEES A CONSISTENT VERSION OF THE TABLE WITHOUT INTERFERING WITH OTHERS.​
* D — DURABILITY​
COMMITTED CHANGES ARE PERMANENTLY STORED IN THE _DELTA_LOG AND UNDERLYING STORAGE, GUARANTEEING RECOVERY EVEN AFTER FAILURES.​


## Delta Lake Architecture

Delta Lake sits as an **open-source storage layer** on top of your existing data lake (cloud object storage such as ADLS, S3, or GCS). It brings reliability, performance, and governance to data lakes.

### Key Architectural Components

| Component | Description |
|---|---|
| **Parquet Data Files** | The actual data is stored as Parquet files in cloud object storage, providing efficient columnar compression and encoding. |
| **Delta Transaction Log (`_delta_log/`)** | A sequentially ordered record of every transaction (commit) ever performed on the table. This is the single source of truth for the table's state. |
| **Checkpoint Files** | Periodic Parquet-format snapshots of the entire log state (every 10 commits by default), enabling fast log replay without reading all individual JSON commits. |
| **Deletion Vectors** | Lightweight `.bin` sidecar files that mark individual rows as deleted, avoiding expensive full-file rewrites for UPDATE/DELETE operations. |




### Write Path
1. **Write new Parquet data files** to cloud storage.
2. **Attempt to commit** a new JSON file to `_delta_log/` containing `add` and `remove` actions.
3. **Optimistic concurrency control** — if another writer committed first, Delta checks for conflicts; if none, it retries the commit with the next version number.
4. **Checkpoint** — every 10 commits, a checkpoint file is written to accelerate future reads.

### Delta Lake vs. Plain Data Lake

| Capability | Plain Data Lake | Delta Lake |
|---|---|---|
| ACID Transactions | ✗ | ✓ |
| Schema Enforcement | ✗ | ✓ |
| Time Travel | ✗ | ✓ (via versioned commits) |
| Scalable Metadata | ✗ | ✓ (transaction log + checkpoints) |
| Unified Batch & Streaming | ✗ | ✓ |
| Data Skipping | ✗ | ✓ (file-level statistics) |
| UPDATE / DELETE / MERGE | ✗ | ✓ |

##Columnar Storage 
* Data inside parquet file stored in columnar format unlike row based traditional database storage
* Provides better compression
* Avoid reading entire columns when only few columns are needed
<img src="/Volumes/training/bronze/images/Screenshot 2026-10-17 at 16.40.52.png" alt="Data Lake Sources" width="800"/>

## Setup 
Create schema bronze_<yourname> 
syntax:
CREATE SCHEMA IF NOT EXISTS training.bronze_<yourname>
MANAGED LOCATION 'abfss://training@unityms.dfs.core.windows.net/bronze<yourname>';
Set the working catalog and schema. 

In [0]:
%sql
USE CATALOG training;

In [0]:

%sql USE SCHEMA bronze;
 SELECT current_catalog() AS catalog, current_schema() AS schema;

## 1. Creating Delta Tables and Inserting Records
**CTAS** (`CREATE TABLE ... AS SELECT`) copies data from the shared catalog into our own managed Delta table. We load 1992–1997 first, then `INSERT` 1998 to create a second version.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS tpc_orders
COMMENT 'Training copy of samples.tpch.orders'
AS
SELECT * FROM samples.tpch.orders
WHERE year(o_orderdate) <= 1997;

## Delta Log

* Deltalog or the trasaction log is heart of the system
* It contains the information about files created or removed during the write
* Contains min,max,null values for first 32 columns of each data files 
* Once commit is successful a new deltalog file will be created with new version number 

Check the storage location
https://portal.azure.com/#@one-consultants.com/resource/subscriptions/11ebd399-8f6c-46af-b26f-71fc4572151b/resourceGroups/ONE-Azure/providers/Microsoft.Storage/storageAccounts/unityms/storagebrowser 

In [0]:
%sql
DESCRIBE history tpc_orders;

In [0]:
%sql
INSERT INTO tpc_orders
SELECT * FROM samples.tpch.orders
WHERE year(o_orderdate) = 1998;

In [0]:
%sql describe detail tpc_orders

## 2. UPDATE, DELETE and MERGE (Immutablity)
Every DML statement is an ACID transaction that creates a **new table version**. Delta never edits files in place: it writes new files (or deletion vectors) and records in the transaction log which files are added and removed.

In [0]:
%sql
-- Escalate large pending orders to URGENT
UPDATE tpc_orders
SET o_orderpriority = '1-URGENT'
WHERE o_orderstatus = 'P' AND o_totalprice > 300000;

In [0]:
%sql
-- Remove fulfilled orders from Q1 1992
DELETE FROM tpc_orders
WHERE o_orderstatus = 'F' AND o_orderdate < '1992-04-01';

In [0]:
%sql
-- Source: recent orders (exist -> UPDATE) + the first week of 1992 fulfilled orders (deleted above -> INSERT)
MERGE INTO tpc_orders AS t
USING (
  SELECT * FROM samples.tpch.orders WHERE o_orderdate >= '1998-07-15'
  UNION ALL
  SELECT * FROM samples.tpch.orders
  WHERE o_orderstatus = 'F' AND o_orderdate BETWEEN '1992-01-01' AND '1992-01-07'
) AS s
ON t.o_orderkey = s.o_orderkey
WHEN MATCHED THEN UPDATE SET t.o_comment = 'updated by merge'
WHEN NOT MATCHED THEN INSERT *;

### How `SELECT * FROM tpc_orders` Works Internally (with Multiple Versions)

<img src="/Volumes/training/bronze/images/Screenshot 2026-10-20 at 13.26.01.png" width="800"/>
When you run a simple `SELECT * FROM tpc_orders`, Delta Lake performs several steps **before** any data is read:


```
SELECT * FROM tpc_orders
        │
        ▼
┌─────────────────────────────┐
│  1. Find _delta_log/        │
│  2. Find latest version     │
│     (highest NNN.json = v3) │
│  3. Replay v0.json →v3.json │
│     (add / remove actions)  │
│  4. Build active file list   │
│     (add – remove = current)│
│  5. Skip files via min/max   │
│     stats (if WHERE)        │
│  6. Read active Parquet files│
│     + apply deletion vectors │
│  7. Return results ✅        │
└─────────────────────────────┘
```

#### Key Takeaway
Even though the table folder may contain **many Parquet files from all past versions** (old files from before UPDATEs, DELETEs, and MERGEs), the transaction log ensures that only the files belonging to the **current version** are read. Old files remain on disk solely to support **time travel** — until `VACUUM` permanently removes them.

## 3. Storage Layout, Delta Log and Checkpoints
### 3.1 Storage layout and versions
A Delta table folder looks like this:



* Each commit JSON lists `add` / `remove` file actions plus metadata.
* Old Parquet files stay in storage after UPDATE/DELETE — that is what makes time travel possible — until `VACUUM` removes them.
* With **deletion vectors** (default on serverless), UPDATE/DELETE mark rows as deleted in a small `.bin` file instead of rewriting whole Parquet files.



In [0]:
%sql
-- One row per version: operation, parameters and operationMetrics (files added/removed, rows affected)
DESCRIBE HISTORY tpc_orders;

In [0]:
%sql
-- Compare numFiles / sizeInBytes with version 0
DESCRIBE DETAIL tpc_orders;

In [0]:
%sql
-- Compare numFiles / sizeInBytes with version 0
DESCRIBE EXTENDED tpc_orders;

In [0]:
%sql
-- How many Parquet files make up each version's snapshot
SELECT 0 AS version, 'CTAS'   AS operation, COUNT(DISTINCT _metadata.file_path) AS data_files, COUNT(*) AS row_count FROM tpc_orders VERSION AS OF 0
UNION ALL SELECT 1, 'INSERT', COUNT(DISTINCT _metadata.file_path), COUNT(*) FROM tpc_orders VERSION AS OF 1
UNION ALL SELECT 2, 'UPDATE', COUNT(DISTINCT _metadata.file_path), COUNT(*) FROM tpc_orders VERSION AS OF 2
UNION ALL SELECT 3, 'DELETE', COUNT(DISTINCT _metadata.file_path), COUNT(*) FROM tpc_orders VERSION AS OF 3
UNION ALL SELECT 4, 'MERGE',  COUNT(DISTINCT _metadata.file_path), COUNT(*) FROM tpc_orders VERSION AS OF 4
ORDER BY version;

In [0]:
%sql
-- Files present in v1 but not in v0 = the new Parquet files written by INSERT
WITH v0 AS (SELECT DISTINCT _metadata.file_name AS f FROM tpc_orders VERSION AS OF 0),
     v1 AS (SELECT _metadata.file_name AS f, _metadata.file_size AS size_bytes, COUNT(*) AS row_count
            FROM tpc_orders VERSION AS OF 1 GROUP BY ALL)
SELECT v1.* FROM v1 LEFT ANTI JOIN v0 USING (f);

In [0]:
%sql
-- Physical files backing the current version (full storage path, size, rows)
SELECT _metadata.file_path AS file_path,
       _metadata.file_size AS size_bytes,
       COUNT(*)            AS row_count
FROM tpc_orders
GROUP BY ALL
ORDER BY file_path;

### 3.2 Checkpoints in the Delta Log
Replaying every JSON commit from version 0 becomes expensive as versions grow. To solve this, Delta Lake writes a **checkpoint** file — a compact Parquet snapshot of the entire table state at a given version.

**What a checkpoint contains:**
- All currently active `add` file actions (the complete set of live Parquet data files)
- Table metadata (schema, partition columns, table properties)


**How checkpoints speed up reads:**
- Instead of replaying `v0.json → v0.json → … → vN.json`, a reader loads the **latest checkpoint** and replays only the JSON commits *after* it.
- Example: if a checkpoint exists at version 10, reading version 12 means loading `checkpoint 10` + replaying `11.json` + `12.json` — just 3 files instead of 13.

**When checkpoints are written:**
- By default, every **10** commits (`delta.checkpointInterval` table property).
- Databricks may also write checkpoints adaptively based on log size.
- You can customise the interval: `ALTER TABLE t SET TBLPROPERTIES ('delta.checkpointInterval' = '5');`

**The `_last_checkpoint` file:**
- A small JSON file in `_delta_log/` that tells readers which checkpoint is the newest, so they don't have to list the directory.



In [0]:
%sql
-- Tiny table; checkpoint every 5 commits instead of the default 10
CREATE TABLE IF NOT EXISTS tpc_log_demo (id INT, note STRING)
TBLPROPERTIES ('delta.checkpointInterval' = '5');

In [0]:
%sql
-- Each INSERT is one commit -> one new JSON file in _delta_log
BEGIN
  DECLARE i INT DEFAULT 1;
  WHILE i <= 12 DO
    INSERT INTO tpc_log_demo VALUES (i, concat('commit ', i));
    SET i = i + 1;
  END WHILE;
END;

In [0]:
%sql
-- One row per commit JSON file; isBlindAppend / operationMetrics come from commitInfo
DESCRIBE HISTORY tpc_log_demo;

In [0]:
%sql
SHOW TBLPROPERTIES tpc_log_demo;

Check the storage location https://portal.azure.com/#@one-consultants.com/resource/subscriptions/11ebd399-8f6c-46af-b26f-71fc4572151b/resourceGroups/ONE-Azure/providers/Microsoft.Storage/storageAccounts/unityms/storagebrowser

## 4. Time Travel
Query any earlier version with `VERSION AS OF n` or `TIMESTAMP AS OF '<ts>'` (use a timestamp from `DESCRIBE HISTORY`). `RESTORE TABLE ... TO VERSION AS OF n` rolls a table back by writing a new version.

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM tpc_orders VERSION AS OF 0) AS v0_rows,
  (SELECT COUNT(*) FROM tpc_orders VERSION AS OF 1) AS v1_rows,
  (SELECT COUNT(*) FROM tpc_orders VERSION AS OF 3) AS v3_rows_after_delete,
  (SELECT COUNT(*) FROM tpc_orders)                 AS current_rows;

In [0]:
%sql
-- Rows whose priority changed between v1 (before UPDATE) and v2 (after UPDATE)
SELECT b.o_orderkey, b.o_totalprice,
       b.o_orderpriority AS priority_v1,
       a.o_orderpriority AS priority_v2
FROM tpc_orders VERSION AS OF 1 b
JOIN tpc_orders VERSION AS OF 2 a USING (o_orderkey)
WHERE b.o_orderpriority <> a.o_orderpriority
LIMIT 20;

In [0]:
%sql
-- Replace the timestamp with one from DESCRIBE HISTORY, then uncomment:
-- SELECT COUNT(*) FROM tpc_orders TIMESTAMP AS OF '2026-10-16T10:00:00';
-- RESTORE TABLE tpc_orders TO VERSION AS OF 1;

-- Side-by-side: the same orders in v1 vs the current version (after MERGE)
SELECT 'v1' AS version, o_orderkey, o_orderstatus, o_comment FROM tpc_orders VERSION AS OF 1 WHERE o_orderdate = '1998-08-02'
UNION ALL
SELECT 'current', o_orderkey, o_orderstatus, o_comment FROM tpc_orders WHERE o_orderdate = '1998-08-02'
ORDER BY o_orderkey, version
LIMIT 10;

In [0]:
%sql
-- table@vN is shorthand for VERSION AS OF N
SELECT 'v0' AS version, COUNT(*) AS row_count FROM tpc_orders@v0
UNION ALL
SELECT 'v3', COUNT(*) FROM tpc_orders@v3;

**How far back can you travel?** A version is readable only while both its commit JSON (kept for `delta.logRetentionDuration`, default 30 days) and its data files (kept until VACUUM removes them after `delta.deletedFileRetentionDuration`, default 7 days) still exist. `RESTORE` is demonstrated on the CDF table in the next section.

## 5. Change Data Feed (CDF)
With `delta.enableChangeDataFeed = true`, Delta records **row-level changes** for every commit. Downstream jobs can read only what changed instead of rescanning the table.

`table_changes('table', start_version [, end_version])` returns the changed rows plus three metadata columns:

| Column | Values |
|---|---|
| `_change_type` | `insert`, `delete`, `update_preimage`, `update_postimage` |
| `_commit_version` | Version that produced the change |
| `_commit_timestamp` | Commit time |

Change data is stored in `_change_data/` (or derived from deletion vectors / row tracking) and is subject to the same VACUUM retention as data files.

In [0]:
%sql
-- Version 0: July 1998 orders, CDF enabled at creation
CREATE TABLE IF NOT EXISTS tpc_orders_cdf
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
AS
SELECT * FROM samples.tpch.orders
WHERE o_orderdate >= '1998-07-01';

In [0]:
%sql
-- v1: UPDATE, v2: DELETE, v3: INSERT
UPDATE tpc_orders_cdf SET o_orderstatus = 'F' WHERE o_orderdate = '1998-07-01';
DELETE FROM tpc_orders_cdf WHERE o_orderdate = '1998-07-02';
INSERT INTO tpc_orders_cdf SELECT * FROM samples.tpch.orders WHERE o_orderdate = '1998-06-30';
DESCRIBE HISTORY tpc_orders_cdf;

In [0]:
%sql
-- CDF change summary by version and change type
SELECT _commit_version, _change_type, COUNT(*) AS rows_changed
FROM table_changes('tpc_orders_cdf', 11, 15)
GROUP BY ALL
ORDER BY _commit_version, _change_type;

-- WHERE IS CDF DATA STORED?
-- CDF data lives in a dedicated `_change_data/` subfolder inside the table's storage directory,
-- NOT in the main data files and NOT in the Delta Log (_delta_log/) JSON/checkpoint files.
--
-- Storage layout:
--   <table_path>/
--   ├── _delta_log/          ← commit JSON + checkpoints (metadata only, NO change rows)
--   ├── part-00000-...parquet ← regular data files (current table state)
--   └── _change_data/        ← CDF Parquet files with _change_type, _commit_version, _commit_timestamp
--
-- These _change_data files are subject to VACUUM just like regular data files,
-- governed by delta.deletedFileRetentionDuration (default 7 days).

In [0]:
%sql
SELECT _commit_version, _change_type, *
FROM table_changes('tpc_orders_cdf', 11, 15) where o_orderdate = '1998-07-02'


In [0]:
%sql
-- Each UPDATE produces a before (preimage) and after (postimage) row
SELECT o_orderkey, o_orderstatus, _change_type, _commit_version, _commit_timestamp
FROM table_changes('tpc_orders_cdf', 10, 11)
ORDER BY o_orderkey, _change_type DESC
LIMIT 10;

In [0]:
%sql
-- Typical downstream pattern: keep only the last change per key (ready for MERGE into a target)
SELECT * EXCEPT (rn)
FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY o_orderkey ORDER BY _commit_version DESC) AS rn
  FROM table_changes('tpc_orders_cdf', 11,13)
  WHERE _change_type <> 'update_preimage'
)
WHERE rn = 1
LIMIT 10;

In [0]:
%sql
-- Roll the CDF table back to its original load; RESTORE writes a NEW version (history is kept)
RESTORE TABLE tpc_orders_cdf TO VERSION AS OF 0;

## Managed vs External Tables

**Managed table**
* Unity Catalog manages both the metadata **and** the data files.
* Data is stored in the managed storage location of the catalog / schema (e.g. `abfss://training@unityms.dfs.core.windows.net/...__unitystorage/...`).
* `DROP TABLE` removes the metadata **and** the data files (recoverable with `UNDROP` within 7 days).

**External table**
* Unity Catalog manages only the metadata; the data files stay in a path **you** control.
* Data is stored in a registered external location (ADLS path).
* Created with a `LOCATION` clause: `CREATE TABLE t (...) LOCATION 'abfss://<container>@<account>.dfs.core.windows.net/<path>'`.
* `DROP TABLE` removes only the metadata; the files remain in storage.
* Use when data must be shared with other tools / systems or already exists in the lake.

| | Managed | External |
|---|---|---|
| Metadata | Unity Catalog | Unity Catalog |
| Data location | Catalog / schema managed storage | Your external location |
| `LOCATION` clause | Not used | Required |
| `DROP TABLE` | Deletes metadata + data | Deletes metadata only |
| Predictive optimization | Yes | No |

All working tables in this notebook (`tpc_orders`, `tpc_orders_cdf`, ...) are **managed** tables. Check with `DESCRIBE EXTENDED <table>` (row `Type`)

#Syntax: CREATE TABLE 'TABLE_NAME' (...) LOCATION 'storage location path'.

#Practial Scenario:
* The data is already maintained in some other datalake before databricks. In Scania we have a scecario where our datafiles are present outside the our cloud storage(on cloud data lake), here we should not have control over droping the table data when table is dropped
* if the organization needs strict decoupling of storage and processing 

#Important thing to remember 
Schema can also be created with location keyword, in such case the tables created underthis (without location keyword) will still be treated as managed table, only when you create a table with keyword location it will be treated as external table


## 6. Constraints
| Type | Enforced? | Purpose |
|---|---|---|
| `NOT NULL` | Yes | Reject NULLs in a column |
| `CHECK (expr)` | Yes | Reject rows where the expression is false; stored as table property `delta.constraints.<name>` |
| `PRIMARY KEY` / `FOREIGN KEY` | No (informational) | Document relationships for tools and BI; with `RELY` the optimizer may use them to remove joins / aggregations |

A write that violates an enforced constraint fails as a whole — no partial data is committed.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS tpc_customer AS
SELECT * FROM samples.tpch.customer;

In [0]:
%sql
-- Re-runnable: drop constraints first if they already exist
ALTER TABLE tpc_orders_cdf DROP CONSTRAINT IF EXISTS fk_orders_customer;
ALTER TABLE tpc_customer   DROP CONSTRAINT IF EXISTS pk_customer;

ALTER TABLE tpc_customer ALTER COLUMN c_custkey SET NOT NULL;
ALTER TABLE tpc_customer ADD CONSTRAINT pk_customer PRIMARY KEY (c_custkey) RELY;

ALTER TABLE tpc_orders_cdf ALTER COLUMN o_orderkey SET NOT NULL;
ALTER TABLE tpc_orders_cdf ADD CONSTRAINT fk_orders_customer
  FOREIGN KEY (o_custkey) REFERENCES tpc_customer (c_custkey);

SELECT table_name, constraint_name, constraint_type
FROM information_schema.table_constraints
WHERE table_schema = 'bronze' AND table_name IN ('tpc_customer', 'tpc_orders_cdf');

In [0]:
%sql
ALTER TABLE tpc_orders_cdf DROP CONSTRAINT IF EXISTS chk_price;
ALTER TABLE tpc_orders_cdf DROP CONSTRAINT IF EXISTS chk_status;

-- Existing rows are validated when a CHECK constraint is added
ALTER TABLE tpc_orders_cdf ADD CONSTRAINT chk_price  CHECK (o_totalprice > 0);
ALTER TABLE tpc_orders_cdf ADD CONSTRAINT chk_status CHECK (o_orderstatus IN ('F', 'O', 'P'));

-- CHECK constraints are stored as delta.constraints.* table properties
SHOW TBLPROPERTIES tpc_orders_cdf;

In [0]:
%sql
-- Try to insert an invalid row (negative price, unknown status); the error is caught and displayed
BEGIN
  DECLARE EXIT HANDLER FOR SQLEXCEPTION
  BEGIN
    DECLARE msg STRING;
    GET DIAGNOSTICS CONDITION 1 msg = MESSAGE_TEXT;
    SELECT 'INSERT rejected' AS result, msg AS error_message;
  END;
  INSERT INTO tpc_orders_cdf VALUES
    (99999999, 1, 'X', -10.00, DATE'1998-08-01', '1-URGENT', 'Clerk#000000001', 0, 'invalid row');
  SELECT 'INSERT succeeded' AS result, NULL AS error_message;
END;

In [0]:
%sql
-- PRIMARY KEY is informational only — duplicates are NOT rejected
INSERT INTO tpc_customer
SELECT * FROM tpc_customer WHERE c_custkey = 1;

-- Proof: c_custkey = 1 now appears twice
SELECT *
FROM tpc_customer
WHERE c_custkey = 1


## 7. Indexes and Data Skipping
 It avoids reading data using:

1. **File statistics (built-in index)** — min/max/null counts per file stored in each `add` action. By default collected for the first 32 columns (`delta.dataSkippingNumIndexedCols`) or for an explicit list (`delta.dataSkippingStatsColumns`). Filters skip files whose range cannot match.
2. **Layout techniques** — Z-order and Liquid Clustering make file statistics far more selective (section 8).
3. **Optimizer statistics** — `ANALYZE TABLE ... COMPUTE STATISTICS` gives the cost-based optimizer row counts and column distributions for join planning.

In [0]:
%sql
-- Collect file statistics only on columns used in filters, then recompute stats for existing files
ALTER TABLE tpc_orders_cdf SET TBLPROPERTIES (
  'delta.dataSkippingStatsColumns' = 'o_orderkey,o_custkey,o_orderdate,o_totalprice,o_orderstatus'
);
ANALYZE TABLE tpc_orders_cdf COMPUTE DELTA STATISTICS;

In [0]:
%sql
ANALYZE TABLE tpc_orders_cdf COMPUTE STATISTICS FOR ALL COLUMNS;
-- Column-level stats: min, max, distinct_count, num_nulls
DESCRIBE EXTENDED tpc_orders_cdf o_custkey;

## 8. Performance Techniques
Delta offers several complementary ways to make queries read less data and write well-sized files:
8.1 Partitioning, 8.2 Z-Ordering, 8.3 Liquid Clustering, 8.4 Compaction, 8.5 Optimized Writes & Auto Compaction, 8.6 Predictive Optimization, 8.7 Key Table Properties.

### 8.1 Partitioning
`PARTITIONED BY` physically splits data into one folder per value (`order_year=1995/part-...parquet`). Queries filtering on the partition column read only matching folders (**partition pruning**).

Use only for low-cardinality columns with large partitions (≥ 1 GB each). High-cardinality partitioning creates many small files — for most new tables prefer Liquid Clustering.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS tpc_orders_partitioned
PARTITIONED BY (order_year)
AS
SELECT *, year(o_orderdate) AS order_year
FROM samples.tpch.orders;

In [0]:
%sql
SHOW PARTITIONS tpc_orders_partitioned;

In [0]:
%sql
-- Note the order_year=YYYY folder in each file path
SELECT order_year,
       COUNT(DISTINCT _metadata.file_path) AS files,
       COUNT(*)                            AS row_count,
       MIN(_metadata.file_path)            AS sample_file_path
FROM tpc_orders_partitioned
GROUP BY order_year
ORDER BY order_year;

In [0]:
%sql
-- Look for PartitionFilters: [order_year = 1995] in the scan node
EXPLAIN FORMATTED
SELECT o_orderpriority, COUNT(*), SUM(o_totalprice)
FROM tpc_orders_partitioned
WHERE order_year = 1995
GROUP BY o_orderpriority;

### 8.2 Z-Ordering
`OPTIMIZE ... ZORDER BY (col)` compacts small files and co-locates similar values of `col` in the same files. Delta keeps min/max statistics per file, so a filter like `o_custkey = 12345` can skip files whose range does not contain the value (**data skipping**).

Below, compare each file's `o_custkey` range before and after Z-ordering: before, every file spans almost the full range; after, ranges are narrow.

In [0]:
%sql
SELECT _metadata.file_name AS file_name, COUNT(*) AS row_count,
       MIN(o_custkey) AS min_custkey, MAX(o_custkey) AS max_custkey
FROM tpc_orders
GROUP BY ALL
ORDER BY min_custkey;

In [0]:
%sql
OPTIMIZE tpc_orders ZORDER BY (o_custkey);

In [0]:
%sql
SELECT _metadata.file_name AS file_name, COUNT(*) AS row_count,
       MIN(o_custkey) AS min_custkey, MAX(o_custkey) AS max_custkey
FROM tpc_orders
GROUP BY ALL
ORDER BY min_custkey;

### 8.3 Liquid Clustering
`CLUSTER BY` replaces both partitioning and Z-order for new tables:
* No folder-per-value; avoids small-file problems on high-cardinality keys.
* Clustering is incremental — `OPTIMIZE` only reclusters new data.
* Keys can be changed later with `ALTER TABLE ... CLUSTER BY` without rewriting the table.
* `CLUSTER BY AUTO` lets Databricks pick keys from the query workload (requires predictive optimization).

Not compatible with `PARTITIONED BY` or `ZORDER` on the same table.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS tpc_lineitems_clustered
CLUSTER BY (l_orderkey)
AS
SELECT * from samples.tpch_sf100.lineitem;

In [0]:
%sql

SELECT _metadata.file_name AS file_name, COUNT(*) AS row_count,
       MIN(l_orderkey) AS min_l_orderkey, MAX(l_orderkey) AS max_l_orderkey
FROM tpc_lineitems_clustered
GROUP BY file_name
ORDER BY min_l_orderkey;

In [0]:
%sql explain select * from tpc_lineitems_clustered where l_orderkey in (9130853)

In [0]:
%sql select * from tpc_lineitems_clustered where l_orderkey in (9130853,1899877)

--Check performance

In [0]:
%sql

-- Let Databricks choose keys automatically
ALTER TABLE tpc_orders_clustered CLUSTER BY AUTO;
DESCRIBE HISTORY tpc_orders_clustered;

#### Comparison: Partitioning vs Z-Order vs Liquid Clustering
| Feature | Partitioning | Z-Order | Liquid Clustering |
|---|---|---|---|
| Physical layout | Folder per value | Sorted within files | Clustered files, no folders |
| Best for | Low-cardinality, very large tables | Filter columns on existing tables | Default choice for new tables |
| Change keys | Full rewrite | Re-run OPTIMIZE | `ALTER TABLE CLUSTER BY` |
| Incremental | n/a | No (rewrites) | Yes |
| Combine with others | With Z-order | With partitioning | Standalone only |

#### When to choose which

**Choose Liquid Clustering when** (recommended default for new tables)
* You are creating a new table, or can recreate an existing one.
* Filter columns are high-cardinality (e.g. `o_custkey`, `o_orderkey`) or you filter on several columns together (`o_custkey`, `o_orderdate`).
* Query patterns may change over time, so you want to change keys with `ALTER TABLE ... CLUSTER BY` without rewriting data.
* The table receives frequent incremental writes (streaming, MERGE, concurrent writers) and you want `OPTIMIZE` to recluster only new data.
* You do not know the best keys yet: use `CLUSTER BY AUTO` with predictive optimization.

**Choose Partitioning when**
* The table is very large (roughly 1 TB+) and each partition still holds at least about 1 GB of data.
* The column is low-cardinality and stable (e.g. date, year, region, country), never high-cardinality IDs.
* You need partition-level operations: dropping/overwriting whole partitions (`REPLACE WHERE`), data lifecycle by date, or physical isolation for governance.
* An existing pipeline or external reader depends on the folder layout.
* Avoid it on small or medium tables: over-partitioning creates many tiny files and slows queries.

**Choose Z-Order when**
* You have an existing (often partitioned) table that you cannot migrate to liquid clustering.
* Queries filter on 1 to 4 high-cardinality columns *within* partitions (e.g. partition by year, Z-order by `o_custkey`).
* Data is mostly written in batches and you can afford periodic full `OPTIMIZE ... ZORDER BY` rewrites.

**Quick decision guide**

| Situation | Recommendation |
|---|---|
| New table, any size | Liquid Clustering (`CLUSTER BY` or `CLUSTER BY AUTO`) |
| Filters on high-cardinality columns | Liquid Clustering (or Z-Order on legacy tables) |
| Huge table, low-cardinality date/region filter, partition-level deletes or overwrites | Partitioning (optionally + Z-Order) |
| Legacy partitioned table, slow filters on other columns | Add Z-Order, or migrate to Liquid Clustering |
| Table under ~1 TB | Do not partition; use Liquid Clustering |
| Unknown or changing query patterns | `CLUSTER BY AUTO` |

### 8.4 Compaction (bin-packing) and the small-file problem
Frequent small writes (streaming, many small batches, over-partitioned jobs) produce thousands of tiny files. Every file means an extra metadata entry, an extra cloud storage request and extra per-file overhead for queries.

`OPTIMIZE table` (without ZORDER) **bin-packs** small files into files of roughly `delta.targetFileSize` (auto-tuned, typically 128 MB–1 GB). It is idempotent, transactional, and only rewrites files that need it. Below we deliberately create 200 small files, then compact them.

In [0]:
%sql
-- Optimized writes / auto compaction OFF, and REPARTITION(200) forces ~200 output files
CREATE TABLE IF NOT EXISTS tpc_orders_small
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'false',
  'delta.autoOptimize.autoCompact'   = 'false'
)
AS
SELECT /*+ REPARTITION(200) */ *
FROM samples.tpch.orders
WHERE year(o_orderdate) = 1995;

In [0]:
%sql
SELECT COUNT(*)                         AS num_files,
       ROUND(AVG(size_bytes) / 1048576, 2) AS avg_file_mb,
       ROUND(SUM(size_bytes) / 1048576, 1) AS total_mb
FROM (SELECT DISTINCT _metadata.file_path, _metadata.file_size AS size_bytes FROM tpc_orders_small);

In [0]:
%sql
-- metrics.numFilesRemoved / numFilesAdded show the effect
OPTIMIZE tpc_orders_small;

In [0]:
%sql
SELECT COUNT(*)                         AS num_files,
       ROUND(AVG(size_bytes) / 1048576, 2) AS avg_file_mb,
       ROUND(SUM(size_bytes) / 1048576, 1) AS total_mb
FROM (SELECT DISTINCT _metadata.file_path, _metadata.file_size AS size_bytes FROM tpc_orders_small);

### 8.5 Optimized Writes and Auto Compaction
Rather than fixing small files afterwards, prevent them at write time:

| Property | What it does |
|---|---|
| `delta.autoOptimize.optimizeWrite = true` | **Optimized writes**: shuffles data before writing so each partition/task writes fewer, larger files (~128 MB). Small extra shuffle cost at write, much better reads. |
| `delta.autoOptimize.autoCompact = auto` | **Auto compaction**: after a write commits, synchronously compacts small files in the affected partitions/table when enough small files accumulate. |

Both are on by default for many operations on Databricks (MERGE, UPDATE, DELETE, CTAS, serverless). We now turn them on for `tpc_orders_small` and repeat the same 200-way write.

In [0]:
%sql
ALTER TABLE tpc_orders_small SET TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'auto'
);

In [0]:
%sql
INSERT INTO tpc_orders_small
SELECT /*+ REPARTITION(200) */ *
FROM samples.tpch.orders
WHERE year(o_orderdate) = 1996;

In [0]:
%sql
-- Compare operationMetrics.numFiles of the CTAS (version 0) with the INSERT after optimized writes
DESCRIBE HISTORY tpc_orders_small;

### 8.6 Predictive Optimization
For **Unity Catalog managed tables**, predictive optimization lets Databricks decide when to run maintenance, based on how each table is written and queried:
* `OPTIMIZE` (compaction and liquid clustering, including choosing keys for `CLUSTER BY AUTO`)
* `VACUUM` (with the table's retention settings)
* `ANALYZE` (statistics for data skipping and the optimizer)

It is enabled at account / metastore, catalog or schema level and inherited downwards:
```sql
ALTER CATALOG one_dbx_dev ENABLE PREDICTIVE OPTIMIZATION;      -- or DISABLE / INHERIT
ALTER SCHEMA  one_dbx_dev.default ENABLE PREDICTIVE OPTIMIZATION;
```
Operations it runs are logged in the system table `system.storage.predictive_optimization_operations_history`. Below we only **check** the current status (no change).

In [0]:
%sql
-- Look for the 'Predictive Optimization' row (e.g. ENABLE inherited from METASTORE)
DESCRIBE SCHEMA EXTENDED training.default;

### 8.7 Key Delta Table Properties
| Property | Default | Purpose |
|---|---|---|
| `delta.targetFileSize` | auto-tuned | Target file size for OPTIMIZE / writes (e.g. `128mb`) |
| `delta.tuneFileSizesForRewrites` | auto | Smaller files for tables with frequent MERGE/UPDATE |
| `delta.autoOptimize.optimizeWrite` | varies | Optimized writes |
| `delta.autoOptimize.autoCompact` | varies | Auto compaction |
| `delta.enableDeletionVectors` | true (new tables) | Soft-delete rows instead of rewriting files on UPDATE/DELETE/MERGE |
| `delta.enableChangeDataFeed` | false | Record row-level changes |
| `delta.enableRowTracking` | true on new UC managed tables | Stable row IDs for incremental processing |
| `delta.checkpointInterval` | 10 | Commits between checkpoints |
| `delta.logRetentionDuration` | interval 30 days | How long commit history is kept |
| `delta.deletedFileRetentionDuration` | interval 7 days | How long removed data files are kept before VACUUM can delete them |
| `delta.dataSkippingNumIndexedCols` / `delta.dataSkippingStatsColumns` | 32 / unset | Columns with file statistics |
| `delta.parquet.compression.codec` | zstd (new managed tables) | Parquet compression |

In [0]:
%sql
ALTER TABLE tpc_orders_small SET TBLPROPERTIES (
  'delta.targetFileSize'               = '128mb',
  'delta.logRetentionDuration'         = 'interval 30 days',
  'delta.deletedFileRetentionDuration' = 'interval 7 days'
);
SHOW TBLPROPERTIES tpc_orders_small;

## 9. VACUUM
* `VACUUM` permanently deletes data files no longer referenced by the current version and older than the retention period (default 7 days). After VACUUM, time travel to versions older than the retention period is no longer possible.
* `DRY RUN` lists files that would be deleted without removing anything. Because our files are only minutes old, the list is expected to be empty.
* `delta.deletedFileRetentionDuration` (default `interval 7 days`) controls how long removed files are kept; `delta.logRetentionDuration` (default `interval 30 days`) controls how long commit history is kept.
* **FULL** mode (default) lists the whole table directory to find unreferenced files. **LITE** mode uses only the transaction log, so it is much faster on large tables but can miss files written by failed jobs.
* Never set retention below the longest-running query or stream on the table; Databricks blocks `RETAIN` values below 7 days unless `spark.databricks.delta.retentionDurationCheck.enabled` is disabled.

In [0]:
%sql
VACUUM tpc_orders DRY RUN;

In [0]:
%sql
-- The 200 small files removed by OPTIMIZE are still in storage (for time travel).
-- They become eligible once older than 168 hours (7 days); today this list is empty.
VACUUM tpc_orders_small RETAIN 168 HOURS DRY RUN;

In [0]:
%sql
-- LITE mode finds candidates from the transaction log only (no directory listing)
VACUUM tpc_orders LITE DRY RUN;

In [0]:
%sql
-- Retention settings that decide what VACUUM and log cleanup may remove
SHOW TBLPROPERTIES tpc_orders_small ('delta.deletedFileRetentionDuration');

In [0]:
%sql
-- Full lifecycle: CTAS -> INSERT -> UPDATE -> DELETE -> MERGE -> OPTIMIZE
DESCRIBE HISTORY tpc_orders;

## 10. Cleanup
Uncomment and run to remove the training objects.

In [0]:
%sql
DROP TABLE IF EXISTS tpc_orders;
DROP TABLE IF EXISTS tpc_orders_partitioned;
DROP TABLE IF EXISTS tpc_orders_clustered;
 DROP TABLE IF EXISTS tpc_log_demo;
 DROP TABLE IF EXISTS tpc_orders_cdf;
 DROP TABLE IF EXISTS tpc_customer;
 DROP TABLE IF EXISTS tpc_orders_small;
 DROP VIEW  IF EXISTS vw_order_summary;
SELECT 'Cleanup statements are commented out' AS note;